**Radiant Typhoon**

In [1]:
import random
from collections import Counter

def hand_is_good(hand, ideal_hands):
    hand_counts = Counter(hand)
    for pattern in ideal_hands:
        pattern_counts = Counter(pattern)
        if all(hand_counts[card] >= need for card, need in pattern_counts.items()):
            return True
    return False

def consistency_with_gamble(
    deckcount,
    ratios,
    names,
    ideal_hands,
    dark_names,          # list of card names that are DARK monsters
    gamble_name='gamble',
    num_hands=100000
):
    # make local copies so we don't mutate your originals
    ratios = ratios.copy()
    names = names.copy()

    # fill deck to deckcount with 'blank' like you do
    blanks = deckcount - sum(ratios)
    if blanks < 0:
        raise ValueError("ratios add up to more than deckcount")
    if blanks > 0:
        ratios.append(blanks)
        names.append('blank')

    # build deck
    deck = []
    j = 0
    for val in ratios:
        for _ in range(val):
            deck.append(names[j])
        j += 1

    base_good = 0          # good before gamble
    gamble_rescues = 0     # bad opener that becomes good thanks to gamble

    for _ in range(num_hands):
        # draw opening hand
        opening_hand = random.sample(deck, 5)

        # 1) normal consistency check
        if hand_is_good(opening_hand, ideal_hands):
            base_good += 1
            continue  # don't let gamble double-count this hand

        # 2) try to use gamble only if it can be used "safely"
        if gamble_name not in opening_hand:
            continue

        # need at least 1 DARK already in hand to avoid punting the whole hand
        if not any(card in dark_names for card in opening_hand):
            continue

        # 3) simulate resolving exactly ONE gamble

        # remaining deck = full deck minus the 5 cards we drew
        remaining_deck = deck.copy()
        for card in opening_hand:
            remaining_deck.remove(card)

        # remove one copy of gamble from hand (we're activating it)
        new_hand = opening_hand.copy()
        new_hand.remove(gamble_name)

        # draw 2 off gamble
        if len(remaining_deck) < 2:
            continue  # extremely rare, safety check
        drawn_two = random.sample(remaining_deck, 2)

        # remove those 2 from remaining_deck (for correctness)
        for card in drawn_two:
            remaining_deck.remove(card)

        new_hand.extend(drawn_two)

        # now we must banish 1 DARK from our hand
        darks_in_hand = [c for c in new_hand if c in dark_names]
        if not darks_in_hand:
            # you wouldn't realistically activate gamble here in real play,
            # but for safety treat as "not rescued"
            continue

        # banish one DARK (for the sim, just remove the first one)
        new_hand.remove(darks_in_hand[0])

        # check if the *post-gamble* hand is now good
        if hand_is_good(new_hand, ideal_hands):
            gamble_rescues += 1

    total_good = base_good + gamble_rescues
    total_consistency = total_good / num_hands
    base_only = base_good / num_hands

    return total_consistency, base_only


In [4]:
deckcount = 40
names = ['vision', '1cc', 'mandate', 'elation', 'mst', 'meghala', 'krosea', 'quickplay', 'fallen']
ratios = [3,        10,       1,       1,         3,        3,           3,          7,        3]

ideal_hands = [

 ['1cc'],
 ['vision', 'meghala'],
 ['vision', 'krosea'],
 ['quickplay', 'mst', 'krosea'],
 ['mst', 'mst', 'krosea'],
 ['mandate', 'mst', 'krosea'],
 ['meghala', 'krosea'],
 ['meghala', 'elation'],
 ['meghala', 'fallen', 'mst'],
 ['meghala', 'mandate', 'mst'],
 ['krosea', 'vision'],
 ['krosea', 'elation'],






    # ... your other combos ...
]

# whatever in your deck is actually DARK / quickplay
dark_names = ['quickplay', 'vision', 'fallen', 'elation']  # example – change this

num_hands = 1000000

total, base = consistency_with_gamble(
    deckcount,
    ratios,
    names,
    ideal_hands,
    dark_names,
    gamble_name='vision',
    num_hands=num_hands
)

print("Base consistency (no gamble):", base)
print("Total consistency (with gamble):", total)
print("gamble adds:", total - base)

NameError: name 'consistency_with_gamble' is not defined

**Dracotail**

In [ ]:
import random
from collections import Counter

def hand_is_good(hand, ideal_hands):
    hand_counts = Counter(hand)
    for pattern in ideal_hands:
        pattern_counts = Counter(pattern)
        if all(hand_counts[card] >= need for card, need in pattern_counts.items()):
            return True
    return False

def consistency(deckcount, ratios, names, ideal_hands, num_hands=100000):
    # make local copies so we don't mutate originals
    ratios = ratios.copy()
    names = names.copy()

    # fill deck to deckcount with blanks
    blanks = deckcount - sum(ratios)
    if blanks < 0:
        raise ValueError("ratios add up to more than deckcount")
    if blanks > 0:
        ratios.append(blanks)
        names.append('blank')

    # build deck
    deck = []
    for name, count in zip(names, ratios):
        deck.extend([name] * count)

    good = 0
    for _ in range(num_hands):
        opening_hand = random.sample(deck, 5)
        if hand_is_good(opening_hand, ideal_hands):
            good += 1

    return good / num_hands

In [ ]:
deckcount = 40
names = ['newalbaz', 'mululu', 'lukias', 'faimena', 'ht', 'cart', 'ecclesia', 'panurg', 'phyrx', 'ketu', 'rahu', 'fusion', 'spoon', 'corwley', 'zoroa']
ratios = [3,             1,       3,        3,       12,    1,       1,            3,      1,     1,        3,        2,          0,      0,          0]

# ONLY 1-card and 2-card “good hands”:
ideal_hands = [

    ['newalbaz'],
    ['ecclesia'],
    ['fusion'],
    ['spoon'],
    ['corwley', 'faimena'],
    ['corwley', 'phryx'],
    ['zoroa', 'faimena'],
    ['zoroa', 'phryx'],
    ['zoroa', 'mululu'],
    ['zoroa', 'panurg'],
    ['cart', 'rahu'],
    ['cart', 'ketu'],
    ['cart', 'mululu'],
    ['cart', 'lukias'],
    ['cart', 'panurg'],
    ['rahu', 'ketu'],
    ['rahu', 'ht'],
    ['rahu', 'mululu'],
    ['rahu', 'lukias'],
    ['rahu', 'faimena'],
    ['rahu', 'panurg'],
    ['rahu', 'phyrx'],
    ['rahu', 'cart'],
    ['rahu', 'crowley'],
    ['rahu', 'zoroa'],
    ['ketu', 'ht'],
    ['ketu', 'mululu'],
    ['ketu', 'lukias'],
    ['ketu', 'faimena'],
    ['ketu', 'panurg'],
    ['ketu', 'phyrx'],
    ['ketu', 'crowley'],
    ['ketu', 'zoroa'],
    ['ht', 'mululu'],
    ['ht', 'lukias'],
    ['mululu', 'lukias'],
    ['mululu', 'faimena'],
    ['mululu', 'panurg'],
    ['mululu', 'phyrx'],
    ['lukias', 'faimena'],
    ['lukias', 'panurg'],
    ['lukias', 'phyrx'],
    ['lukias', 'crowley'],
    ['lukias', 'zoroa'],
    ['faimena', 'panurg', 'phyrx'],
    ['faimena', 'phyrx', 'panurg'],
    ['faimena', 'ht', 'panurg'],
    ['faimena', 'ht', 'phyrx'],
    ['cart', 'ht', 'phyrx'],
    ['cart', 'faimena', 'ht'],
    ['cart', 'faimena', 'faimena'],
    ['cart', 'phyrx', 'faimena'],











]

num_hands = 1_000_000

base = consistency(deckcount, ratios, names, ideal_hands, num_hands=num_hands)
print("Consistency (1-2 card patterns only):", base)


Consistency (1-2 card patterns only): 0.974848
